
# DEP Fall / No-Fall Decision Tree — Pre-labelled CSV Version

Use this notebook with the two CSV files that already contain a column named:

```text
label
```

with values:

```text
FALL
NO_FALL
```

This version does **not** use the old manual `LABELS = {...}` dictionary.

It:
1. uploads your marked CSVs;
2. reads the existing `label` column;
3. creates one feature window per fall `Event_ID`;
4. samples separate NO_FALL windows;
5. trains a Decision Tree;
6. shows accuracy, confusion matrix, feature importance and learned split thresholds;
7. exports the model/results for your website.


In [ ]:

# Cell 1 — Imports

import io
import joblib
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
PRE_EVENT_S = 0.50
POST_EVENT_S = 0.70

print("Ready.")


In [ ]:

# Cell 2 — Upload BOTH marked CSV files

uploaded = files.upload()

csv_files = {
    name: content
    for name, content in uploaded.items()
    if name.lower().endswith(".csv")
}

if not csv_files:
    raise ValueError("Please upload your marked CSV files.")

print("Uploaded:")
for name in csv_files:
    print("-", name)


In [ ]:

# Cell 3 — Load and verify the labels
# This version accepts BOTH:
#   label
#   Fall_Label

EXPECTED_SENSOR_COLUMNS = [
    "Timestamp",
    "Accelerometer_X_g",
    "Accelerometer_Y_g",
    "Accelerometer_Z_g",
    "Gyroscope_X_deg_s",
    "Gyroscope_Y_deg_s",
    "Gyroscope_Z_deg_s",
]

recordings = {}

for filename, content in csv_files.items():
    df = pd.read_csv(io.BytesIO(content))

    # Accept the older marked CSV format automatically.
    if "label" not in df.columns and "Fall_Label" in df.columns:
        df = df.rename(columns={"Fall_Label": "label"})
        print(f"{filename}: renamed Fall_Label -> label")

    required = EXPECTED_SENSOR_COLUMNS + ["label"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"{filename} is missing: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        errors="coerce"
    )

    for col in EXPECTED_SENSOR_COLUMNS[1:]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    df["label"] = (
        df["label"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Clean common accidental text forms.
    df["label"] = df["label"].replace({
        "NO FALL": "NO_FALL",
        "NO-FALL": "NO_FALL",
        "NOFALL": "NO_FALL",
    })

    df = df.dropna(
        subset=EXPECTED_SENSOR_COLUMNS
    ).copy()

    df = (
        df.drop_duplicates()
        .sort_values("Timestamp")
        .reset_index(drop=True)
    )

    df["acc_mag_g"] = np.sqrt(
        df["Accelerometer_X_g"]**2
        + df["Accelerometer_Y_g"]**2
        + df["Accelerometer_Z_g"]**2
    )

    df["gyro_mag_dps"] = np.sqrt(
        df["Gyroscope_X_deg_s"]**2
        + df["Gyroscope_Y_deg_s"]**2
        + df["Gyroscope_Z_deg_s"]**2
    )

    recordings[filename] = df

    print("\n", filename)
    print(df["label"].value_counts(dropna=False))

print("\nLabels were found successfully.")


In [ ]:

# Cell 4 — Feature functions

ACC_COLS = [
    "Accelerometer_X_g",
    "Accelerometer_Y_g",
    "Accelerometer_Z_g",
]

def longest_low_g_duration(local, threshold=0.45):
    local = local.sort_values("Timestamp")

    longest = 0.0
    start = None
    last = None

    for row in local.itertuples():
        if row.acc_mag_g <= threshold:
            if start is None:
                start = row.Timestamp
            last = row.Timestamp
        else:
            if start is not None and last is not None:
                longest = max(
                    longest,
                    (last - start).total_seconds()
                )
            start = None
            last = None

    if start is not None and last is not None:
        longest = max(
            longest,
            (last - start).total_seconds()
        )

    return float(longest)


def tilt_change_deg(local):
    if len(local) < 6:
        return 0.0

    n = max(3, int(len(local) * 0.15))

    start_vec = (
        local[ACC_COLS]
        .head(n)
        .mean()
        .to_numpy(dtype=float)
    )

    end_vec = (
        local[ACC_COLS]
        .tail(n)
        .mean()
        .to_numpy(dtype=float)
    )

    denom = (
        np.linalg.norm(start_vec)
        * np.linalg.norm(end_vec)
    )

    if denom <= 1e-9:
        return 0.0

    cosine = np.clip(
        np.dot(start_vec, end_vec) / denom,
        -1.0,
        1.0
    )

    return float(
        np.degrees(np.arccos(cosine))
    )


def extract_features(df, center):
    start = center - pd.Timedelta(seconds=PRE_EVENT_S)
    end = center + pd.Timedelta(seconds=POST_EVENT_S)

    local = df[
        (df["Timestamp"] >= start)
        & (df["Timestamp"] <= end)
    ].copy()

    if len(local) < 5:
        return None

    return {
        "acc_peak_g": float(local["acc_mag_g"].max()),
        "acc_min_g": float(local["acc_mag_g"].min()),
        "acc_mean_g": float(local["acc_mag_g"].mean()),
        "acc_std_g": float(local["acc_mag_g"].std(ddof=0)),
        "acc_pp_g": float(
            local["acc_mag_g"].max()
            - local["acc_mag_g"].min()
        ),
        "gyro_peak_dps": float(local["gyro_mag_dps"].max()),
        "gyro_mean_dps": float(local["gyro_mag_dps"].mean()),
        "gyro_std_dps": float(local["gyro_mag_dps"].std(ddof=0)),
        "low_g_duration_s": longest_low_g_duration(local, 0.45),
        "tilt_change_deg": tilt_change_deg(local),
    }


In [ ]:

# Cell 5 — Build FALL event feature rows

fall_rows = []
fall_times_by_file = {}

for filename, df in recordings.items():
    fall_df = df[df["label"] == "FALL"].copy()

    if fall_df.empty:
        fall_times_by_file[filename] = []
        continue

    # Your marked files contain Event_ID.
    if "Event_ID" not in df.columns:
        raise ValueError(
            f"{filename} does not contain Event_ID."
        )

    event_ids = (
        fall_df["Event_ID"]
        .dropna()
        .unique()
    )

    fall_times_by_file[filename] = []

    for event_id in event_ids:
        event_rows = fall_df[
            fall_df["Event_ID"] == event_id
        ]

        if event_rows.empty:
            continue

        peak_idx = event_rows["acc_mag_g"].idxmax()
        center = df.loc[peak_idx, "Timestamp"]

        features = extract_features(df, center)

        if features is None:
            continue

        fall_times_by_file[filename].append(center)

        fall_rows.append({
            "recording": filename,
            "event_id": int(float(event_id)),
            "event_timestamp": center,
            "label": "FALL",
            **features,
        })

fall_features = pd.DataFrame(fall_rows)

print("FALL events extracted:", len(fall_features))
display(fall_features)


In [ ]:

# Cell 6 — Build NO_FALL feature rows

# Select stable NO_FALL windows away from every marked fall.
# We take about the same number of NO_FALL examples as FALL examples.

no_fall_candidates = []

for filename, df in recordings.items():
    fall_times = fall_times_by_file.get(filename, [])

    # Candidate centre every ~1.5 seconds.
    if len(df) == 0:
        continue

    start_time = df["Timestamp"].iloc[0]
    end_time = df["Timestamp"].iloc[-1]

    center = start_time + pd.Timedelta(seconds=0.7)

    while center < end_time - pd.Timedelta(seconds=0.7):

        # Keep NO_FALL windows at least 2 seconds away from a fall event.
        too_close = any(
            abs((center - t).total_seconds()) < 2.0
            for t in fall_times
        )

        if not too_close:
            local = df[
                (df["Timestamp"] >= center - pd.Timedelta(seconds=PRE_EVENT_S))
                & (df["Timestamp"] <= center + pd.Timedelta(seconds=POST_EVENT_S))
            ]

            if (
                len(local) >= 5
                and (local["label"] == "NO_FALL").all()
            ):
                features = extract_features(df, center)

                if features is not None:
                    no_fall_candidates.append({
                        "recording": filename,
                        "event_id": None,
                        "event_timestamp": center,
                        "label": "NO_FALL",
                        **features,
                    })

        center += pd.Timedelta(seconds=1.5)


no_fall_candidates = pd.DataFrame(no_fall_candidates)

if no_fall_candidates.empty:
    raise ValueError("No valid NO_FALL windows were found.")

# Sample a balanced number of NO_FALL windows.
target_no_fall = min(
    len(no_fall_candidates),
    max(len(fall_features), 1)
)

no_fall_features = no_fall_candidates.sample(
    n=target_no_fall,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("NO_FALL windows extracted:", len(no_fall_features))
display(no_fall_features)


In [ ]:

# Cell 7 — Final event-level training dataset

features_df = pd.concat(
    [
        fall_features,
        no_fall_features,
    ],
    ignore_index=True,
)

features_df = features_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

display(features_df)

print("\nClass counts:")
print(features_df["label"].value_counts())


In [ ]:

# Cell 8 — Train Decision Tree

FEATURE_COLUMNS = [
    "acc_peak_g",
    "acc_min_g",
    "acc_mean_g",
    "acc_std_g",
    "acc_pp_g",
    "gyro_peak_dps",
    "gyro_mean_dps",
    "gyro_std_dps",
    "low_g_duration_s",
    "tilt_change_deg",
]

X = features_df[FEATURE_COLUMNS]
y = features_df["label"]

if y.nunique() < 2:
    raise ValueError("Need both FALL and NO_FALL examples.")

class_counts = y.value_counts()

if len(features_df) >= 20 and class_counts.min() >= 5:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    evaluation_name = "STRATIFIED 30% HOLDOUT TEST"
else:
    X_train, X_test = X.copy(), X.copy()
    y_train, y_test = y.copy(), y.copy()
    evaluation_name = "TRAINING DATA ONLY"

model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)
balanced_accuracy = balanced_accuracy_score(y_test, pred)

print("Evaluation:", evaluation_name)
print(f"Accuracy: {accuracy:.3f}")
print(f"Balanced accuracy: {balanced_accuracy:.3f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        pred,
        zero_division=0
    )
)


In [ ]:

# Cell 9 — Confusion matrix

ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred
)

plt.title("Fall / No-Fall Confusion Matrix")
plt.show()


In [ ]:

# Cell 10 — Decision Tree

plt.figure(figsize=(22, 10))

plot_tree(
    model,
    feature_names=FEATURE_COLUMNS,
    class_names=[str(x) for x in model.classes_],
    filled=True,
    rounded=True,
    fontsize=10,
)

plt.title("Nesso FALL / NO_FALL Decision Tree")
plt.show()


In [ ]:

# Cell 11 — Learned rules and thresholds

rules_text = export_text(
    model,
    feature_names=FEATURE_COLUMNS
)

print(rules_text)

tree = model.tree_

threshold_rows = []

for node_id in range(tree.node_count):
    feature_index = tree.feature[node_id]

    if feature_index < 0:
        continue

    threshold_rows.append({
        "node_id": node_id,
        "feature": FEATURE_COLUMNS[feature_index],
        "threshold": float(tree.threshold[node_id]),
        "samples_at_node": int(tree.n_node_samples[node_id]),
        "gini": float(tree.impurity[node_id]),
    })

threshold_df = pd.DataFrame(threshold_rows)

print("\nLearned split thresholds:")
display(threshold_df)


In [ ]:

# Cell 12 — Feature importance

importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": model.feature_importances_,
}).sort_values(
    "importance",
    ascending=False
)

display(importance_df)

plt.figure(figsize=(10, 5))
plt.bar(
    importance_df["feature"],
    importance_df["importance"]
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importance")
plt.title("Decision Tree Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:

# Cell 13 — Export results

features_df.to_csv(
    "fall_no_fall_event_features.csv",
    index=False
)

threshold_df.to_csv(
    "fall_no_fall_tree_thresholds.csv",
    index=False
)

importance_df.to_csv(
    "fall_no_fall_feature_importance.csv",
    index=False
)

with open("fall_no_fall_rules.txt", "w") as f:
    f.write(rules_text)

joblib.dump(
    model,
    "fall_no_fall_decision_tree.joblib"
)

with open("model_feature_order.txt", "w") as f:
    for feature in FEATURE_COLUMNS:
        f.write(feature + "\n")

output_files = [
    "fall_no_fall_event_features.csv",
    "fall_no_fall_tree_thresholds.csv",
    "fall_no_fall_feature_importance.csv",
    "fall_no_fall_rules.txt",
    "fall_no_fall_decision_tree.joblib",
    "model_feature_order.txt",
]

with zipfile.ZipFile(
    "DEP_Fall_NoFall_Model_Outputs.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    for filename in output_files:
        z.write(filename)

print("Created DEP_Fall_NoFall_Model_Outputs.zip")

files.download(
    "DEP_Fall_NoFall_Model_Outputs.zip"
)



## What to screenshot for your report

Take screenshots of:

1. Cell 3 — label counts found successfully
2. Cell 7 — event-level FALL / NO_FALL feature table
3. Cell 8 — accuracy and classification report
4. Cell 9 — confusion matrix
5. Cell 10 — Decision Tree
6. Cell 11 — learned split thresholds
7. Cell 12 — feature importance

Do not describe the automatic historical labels as independently verified real-world fall ground truth unless the event labels were actually confirmed during testing.
